In [1]:
import pandas as pd

In [2]:
dataset_full = pd.read_csv("./cars_dataset.csv", encoding="cp1252")

filtered = dataset_full[
    dataset_full["Company Names"].isin(
        [
            "TOYOTA",
            "VOLVO",
            "Toyota",
            "Volvo",
            "BMW",
            "Tesla",
            "Volkswagen",
        ]
    )
]

df_sample = filtered.sample(n=12, random_state=123)

In [9]:
df = df_sample.drop(columns=["Engines", "Fuel Types"])

cols = [
    "CC/Battery Capacity",
    "HorsePower",
    "Total Speed",
    "Performance(0 - 100 )KM/H",
    "Cars Prices",
    "Seats",
    "Torque"
]

df[cols] = df[cols].apply(
    lambda col: col.astype(str)
    .str.replace(',', '')
    .str.extract(r'(\d+\.?\d*)')[0]
).astype(float)

int_cols = cols.copy()
int_cols.remove("Performance(0 - 100 )KM/H")

df[int_cols] = df[int_cols].astype(int)
df.reset_index(drop=True, inplace=True)


In [4]:
df

,Company Names,Cars Names,CC/Battery Capacity,HorsePower,Total Speed,Performance(0 - 100 )KM/H,Cars Prices,Seats,Torque
0,Volkswagen,Golf 4MOTION,1968,150,250,4.9,40000,5,400
1,Volvo,Volvo VNR,12800,400,110,17.0,95000,2,1800
2,TOYOTA,GR COROLLA,1618,300,230,5.0,36995,5,370
3,BMW,M550i XDRIVE,4395,523,250,3.6,78000,5,650
4,TOYOTA,INNOVA HYCROSS,1987,184,165,9.5,40000,7,250
5,Volkswagen,Passat Hybrid,1395,215,210,7.4,32000,5,330
6,Tesla,Model X,100,670,250,3.8,98490,7,967
7,Volkswagen,Jetta Hybrid,1395,170,210,8.6,28000,5,250
8,Volvo,Volvo FMX,13000,500,110,16.5,110000,2,2500
9,Volkswagen,Passat,1984,150,210,7.8,25000,5,250


# 2.1


3 > 6
3 > 2
3 > 0

6 > 2
6 > 0
6 > 10

2 > 10
0 > 10

0 ~ 2

10 > 3 (to create cycle)

In [55]:
from pulp import *

prefs = [
    (3,6), (3,2), (3,0),
    (6,2), (6,0), (6,10),
    (2,10), (0,10),
    (10,3)
]

gain = ['CC/Battery Capacity', 'HorsePower',
       'Total Speed',  'Seats',
       'Torque']

cost = ['Performance(0 - 100 )KM/H', 'Cars Prices']
criteria = gain + cost


In [56]:

def generate_levels(min_val, max_val, gamma):
    return [
        min_val + (j / (gamma - 1)) * (max_val - min_val)
        for j in range(gamma)
    ]

criterion_values = {}
gamma = 4

for c in criteria:
    min_val = df[c].min()
    max_val = df[c].max()
    criterion_values[c] = generate_levels(min_val, max_val, gamma)

In [60]:
def solve():
    prob = LpProblem("UTA", LpMinimize)

    u = {}
    # marginal utilities
    for c in criteria:
        u[c] = {}
        for val in criterion_values[c]:
            u[c][val] = LpVariable(f"u_{c}_{val}", 0, 1)

    # monotonicity
    for c in criteria:
        vals = criterion_values[c]
        for i in range(len(vals)-1):
            if c in cost:
                prob += u[c][vals[i]] >= u[c][vals[i+1]]
            else:
                prob += u[c][vals[i]] <= u[c][vals[i+1]]

    #normalization
    for c in criteria:
        vals = criterion_values[c]
        if c in cost:
            prob += u[c][vals[-1]] == 0
        else:
            prob += u[c][vals[0]] == 0

    # weight constraints
    weights = {}
    for c in criteria:
        vals = criterion_values[c]
        if c in cost:
            weights[c] = u[c][vals[0]]
        else:
            weights[c] = u[c][vals[-1]]
        prob += weights[c] <= 0.5
        prob += weights[c] >= 0.1


    prob += lpSum(weights[c] for c in criteria) == 1

    def marginal_utility(c, val):
        vals = sorted(criterion_values[c])

        for j in range(len(vals) - 1):
            xj = vals[j]
            xj1 = vals[j+1]

            if xj <= val <= xj1:
                return (
                    u[c][xj] +
                    (val - xj) / (xj1 - xj) *
                    (u[c][xj1] - u[c][xj])
                )

        return u[c][vals[-1]]


    def utility(i):
        return lpSum(
            marginal_utility(c, df.loc[i, c])
            for c in criteria
        )

    epsilon = 0.01
    M = 10

    y = {}

    for k, (a,b) in enumerate(prefs):
        y[k] = LpVariable(f"y_{k}", cat="Binary")
        prob += utility(a) >= utility(b) + epsilon - M*y[k]

    prob += lpSum(y[k] for k in y)

    #prob.solve(PULP_CBC_CMD(msg=0))
    prob.solve(GLPK_CMD(path="/opt/homebrew/bin/glpsol", msg=0)) #for mac
    return prob, y, criteria, weights



In [63]:
def print_results(prob, y, criteria, weights):
    print("Status: ", LpStatus[prob.status])

    print("Violated constraints:")
    violated = []
    for k in y:
        if value(y[k]) > 0.5:
            print(f"Removed: {prefs[k]}")
            violated.append(k)

    print("Consistent preference subset:")
    for k in range(len(prefs)):
        if k not in violated:
            print(prefs[k])

    print("Weights:")
    for c in criteria:
        print(c, round(value(weights[c]),3))


In [64]:
res = solve()
print_results(res[0], res[1], res[2], res[3])

Status:  Optimal
Violated constraints:
Removed: (10, 3)
Consistent preference subset:
(3, 6)
(3, 2)
(3, 0)
(6, 2)
(6, 0)
(6, 10)
(2, 10)
(0, 10)
Weights:
CC/Battery Capacity 0.332
HorsePower 0.168
Total Speed 0.1
Seats 0.1
Torque 0.1
Performance(0 - 100 )KM/H 0.1
Cars Prices 0.1
